# Génération Dataset



## Imports

In [1]:
import sys
import pickle
import time
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter
from tqdm import tqdm

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.llm.llm_factory import LLMFactory
from src.simulation_workflow import SimulationWorkflow

print('Imports OK')

Imports OK


## Paramètres

In [2]:
CASES_PER_CLASS = 50   # 50 x 4 = 200 cas total
DELAY = 2              # secondes entre appels API

# 10 symptômes binaires — ordre fixe important (ne pas changer après entraînement)
SYMPTOMES_CLES = [
    "douleur thoracique",
    "dyspnée",
    "perte de connaissance",
    "hémorragie",
    "fracture",
    "fièvre élevée",
    "douleur abdominale",
    "nausée vomissement",
    "symptôme mineur",
    "pas urgence",
]

# Pathologies par classe — inspirées des 79 cas existants
PATHOLOGIES = {
    "ROUGE": [
        "Homme de 65 ans avec infarctus du myocarde",
        "Femme de 58 ans avec AVC ischémique",
        "Homme de 72 ans avec hémorragie digestive massive",
        "Femme de 48 ans avec embolie pulmonaire",
        "Homme de 55 ans avec détresse respiratoire aiguë",
        "Femme de 70 ans avec arrêt cardiaque récupéré",
        "Homme de 40 ans avec polytraumatisme suite accident",
        "Femme de 62 ans avec choc septique",
        "Homme de 50 ans avec douleur thoracique intense irradiant bras gauche",
        "Femme de 45 ans avec paralysie brutale hémicorps droit",
    ],
    "JAUNE": [
        "Femme de 35 ans avec fracture tibia-péroné suite chute",
        "Homme de 42 ans avec appendicite aiguë",
        "Femme de 28 ans avec colique néphrétique",
        "Homme de 50 ans avec pneumonie sévère",
        "Femme de 38 ans avec brûlure 2ème degré étendue",
        "Homme de 60 ans avec crise d'asthme modérée",
        "Femme de 45 ans avec pyélonéphrite avec fièvre à 39.5",
        "Homme de 33 ans avec fracture du poignet suite chute",
        "Femme de 55 ans avec douleur abdominale intense",
        "Homme de 48 ans avec crise hypertensive 180/110",
    ],
    "VERT": [
        "Femme de 30 ans avec gastro-entérite",
        "Homme de 25 ans avec entorse cheville",
        "Femme de 45 ans avec infection urinaire simple",
        "Homme de 32 ans avec otite moyenne aiguë",
        "Femme de 28 ans avec conjonctivite infectieuse",
        "Homme de 38 ans avec migraine sans signe de gravité",
        "Femme de 22 ans avec angine",
        "Homme de 55 ans avec lombalgies aiguës",
        "Femme de 40 ans avec urticaire allergique",
        "Homme de 29 ans avec pharyngite fébrile",
    ],
    "GRIS": [
        "Homme de 22 ans pour certificat médical sport",
        "Femme de 40 ans pour renouvellement ordonnance",
        "Homme de 35 ans avec rhume léger",
        "Femme de 50 ans pour résultats d'analyses",
        "Homme de 28 ans avec petite coupure superficielle",
        "Femme de 33 ans avec courbatures légères",
        "Homme de 45 ans avec fatigue sans autre symptôme",
        "Femme de 27 ans avec nez bouché depuis 3 jours",
        "Homme de 60 ans pour bilan de santé",
        "Femme de 38 ans avec légère douleur de gorge",
    ],
}

print(f'Paramètres OK')
print(f'  {CASES_PER_CLASS} cas x 4 classes = {CASES_PER_CLASS * 4} cas total')
print(f'  {len(SYMPTOMES_CLES)} features symptômes + 8 constantes = 18 features total')

Paramètres OK
  50 cas x 4 classes = 200 cas total
  10 features symptômes + 8 constantes = 18 features total


## Fonctions

In [3]:
def encode_symptomes_binaire(symptomes: list) -> list:
    """
    Encode une liste de symptômes en 10 features binaires.
    Chaque feature vaut 1 si le symptôme est présent, 0 sinon.
    """
    text = " ".join(symptomes).lower() if symptomes else ""
    
    # Mots-clés associés à chaque symptôme cible
    keywords = {
        "douleur thoracique":      ["thoracique", "poitrine", "thorax", "coeur", "cardiaque"],
        "dyspnée":                 ["dyspnée", "respir", "souffle", "essoufflement", "étouffement"],
        "perte de connaissance":   ["connaissance", "syncope", "évanouissement", "inconscient"],
        "hémorragie":              ["hémorragie", "saignement", "sang", "hémoptysie"],
        "fracture":                ["fracture", "cassure", "os", "traumatisme"],
        "fièvre élevée":           ["fièvre", "température", "hyperthermie", "fébril"],
        "douleur abdominale":      ["abdomin", "ventre", "estomac", "intestin", "colique"],
        "nausée vomissement":      ["nausée", "vomissement", "vomit", "nausées"],
        "symptôme mineur":         ["entorse", "otite", "conjonctivite", "migraine", "angine", "pharyngite", "urticaire"],
        "pas urgence":             ["certificat", "ordonnance", "rhume", "résultats", "courbatures", "fatigue", "bilan"],
    }
    
    features = []
    for symptome in SYMPTOMES_CLES:
        kws = keywords[symptome]
        present = int(any(kw in text for kw in kws))
        features.append(present)
    
    return features


def build_features(ml_data: dict) -> list:
    """
    Construit le vecteur de 18 features :
    [FC, FR, SpO2, TA_sys, TA_dia, Temp, Age, Sexe, + 10 symptômes binaires]
    """
    # 8 constantes (valeurs réelles, non normalisées — la normalisation se fait à l'entraînement)
    fc   = ml_data.get('fc')   or 75
    fr   = ml_data.get('fr')   or 16
    spo2 = ml_data.get('spo2') or 98
    ta_s = ml_data.get('ta_systolique')  or 120
    ta_d = ml_data.get('ta_diastolique') or 80
    temp = ml_data.get('temperature')    or 37.0
    age  = ml_data.get('age')  or 40
    sexe = 1 if ml_data.get('sexe') in ['M', 'Homme', 'H'] else 0
    
    constantes = [fc, fr, spo2, ta_s, ta_d, temp, age, sexe]
    
    # 10 symptômes binaires
    symptomes = ml_data.get('symptomes', [])
    symptomes_bin = encode_symptomes_binaire(symptomes)
    
    return constantes + symptomes_bin


print('Fonctions OK')
print(f'  Vecteur features : {len(build_features({}))} dimensions')

Fonctions OK
  Vecteur features : 18 dimensions


## Test encode_symptomes_binaire

In [4]:
# Vérification rapide avant de lancer la génération
test_symptomes = ["douleur thoracique intense", "dyspnée au repos", "nausées"]
encoded = encode_symptomes_binaire(test_symptomes)

print('Test encodage symptômes :')
for s, v in zip(SYMPTOMES_CLES, encoded):
    print(f'  {s:<30} : {v}')

Test encodage symptômes :
  douleur thoracique             : 1
  dyspnée                        : 1
  perte de connaissance          : 0
  hémorragie                     : 0
  fracture                       : 1
  fièvre élevée                  : 0
  douleur abdominale             : 0
  nausée vomissement             : 1
  symptôme mineur                : 0
  pas urgence                    : 0


## Génération des cas

In [5]:
llm = LLMFactory.create("mistral", "mistral-small-latest")
workflow = SimulationWorkflow(llm, max_turns=6)

X_list = []
y_list = []
metadata_list = []
errors = 0

import io
from contextlib import redirect_stdout

for target_class in ["ROUGE", "JAUNE", "VERT", "GRIS"]:
    print(f"\n{'='*60}")
    print(f"Génération classe {target_class}")
    print(f"{'='*60}")
    
    pathologies = PATHOLOGIES[target_class]
    class_count = 0
    
    for i in tqdm(range(CASES_PER_CLASS)):
        pathology = pathologies[i % len(pathologies)]
        
        try:
            with redirect_stdout(io.StringIO()):
                result = workflow.run_simulation(pathology=pathology)
            
            ml_data = workflow.export_for_ml()
            features = build_features(ml_data)
            
            X_list.append(features)
            y_list.append(target_class)
            metadata_list.append({
                'pathology':  ml_data.get('pathology'),
                'age':        ml_data.get('age'),
                'sexe':       ml_data.get('sexe'),
                'symptomes':  ml_data.get('symptomes', []),
                'fc':         ml_data.get('fc'),
                'fr':         ml_data.get('fr'),
                'spo2':       ml_data.get('spo2'),
                'ta_sys':     ml_data.get('ta_systolique'),
                'ta_dia':     ml_data.get('ta_diastolique'),
                'temperature':ml_data.get('temperature'),
                'label':      target_class,
            })
            class_count += 1
            workflow.reset()
            time.sleep(DELAY)
            
        except Exception as e:
            errors += 1
            if "429" in str(e):
                print(f"\nRate limit — pause 30s...")
                time.sleep(30)
            else:
                print(f"\nErreur cas {i}: {e}")
                time.sleep(3)
            workflow.reset()
    
    print(f"{target_class} : {class_count} cas générés")

X = np.array(X_list)
y = np.array(y_list)

print(f"\n{'='*60}")
print(f"GÉNÉRATION TERMINÉE")
print(f"  X shape : {X.shape}")
print(f"  Erreurs : {errors}")
print(f"  Distribution : {Counter(y)}")
print(f"{'='*60}")


Génération classe ROUGE


  4%|▍         | 2/50 [00:39<15:41, 19.61s/it]


Rate limit — pause 30s...


 10%|█         | 5/50 [01:56<17:29, 23.33s/it]


Rate limit — pause 30s...


...


Rate limit — pause 30s...


 98%|█████████▊| 49/50 [21:39<00:27, 27.43s/it]


Rate limit — pause 30s...


100%|██████████| 50/50 [22:25<00:00, 26.91s/it]

GRIS : 34 cas générés

GÉNÉRATION TERMINÉE
  X shape : (129, 18)
  Erreurs : 71
  Distribution : Counter({np.str_('GRIS'): 34, np.str_('ROUGE'): 33, np.str_('JAUNE'): 31, np.str_('VERT'): 31})


## Vérification

In [6]:
print('VÉRIFICATION DATASET v2')
print('='*60)
print(f'Shape X : {X.shape}  (attendu : ~200 x 18)')
print(f'Shape y : {y.shape}')
print()

# Distribution
counts = Counter(y)
print('Distribution des classes :')
for label in ['ROUGE', 'JAUNE', 'VERT', 'GRIS']:
    n = counts.get(label, 0)
    bar = '█' * n
    print(f'  {label:<6} : {n:3d} ({n/len(y)*100:.1f}%) {bar[:40]}')

print()
print('Exemple cas 1 :')
m = metadata_list[0]
print(f'  Pathologie : {m["pathology"]}')
print(f'  Symptômes  : {m["symptomes"]}')
print(f'  FC={m["fc"]}  FR={m["fr"]}  SpO2={m["spo2"]}  Temp={m["temperature"]}')
print(f'  Label      : {m["label"]}')
print()
print('Features cas 1 :')
print(f'  Constantes : {X[0, :8]}')
print(f'  Symptômes  : {X[0, 8:]}')
print(f'  Noms symp  : {[s for s, v in zip(SYMPTOMES_CLES, X[0, 8:]) if v == 1]}')

VÉRIFICATION DATASET v2
Shape X : (129, 18)  (attendu : ~200 x 18)
Shape y : (129,)

Distribution des classes :
  ROUGE  :  33 (25.6%) █████████████████████████████████
  JAUNE  :  31 (24.0%) ███████████████████████████████
  VERT   :  31 (24.0%) ███████████████████████████████
  GRIS   :  34 (26.4%) ██████████████████████████████████

Exemple cas 1 :
  Pathologie : Homme de 65 ans avec infarctus du myocarde
  Symptômes  : ['douleurs dans le dos', 'gêne pour marcher et dormir', 'engourdissements dans les jambes', 'difficultés à contrôler les jambes', 'difficultés à marcher droit', "picotements dans les jambes remontant jusqu'aux fesses", 'douleurs comme des décharges électriques', "fuites d'urine", 'constipation', 'mal aux jambes après quelques minutes debout ou en marchant']
  FC=115  FR=22  SpO2=90  Temp=36.8
  Label      : ROUGE

Features cas 1 :
  Constantes : [115.   22.   90.   90.   65.   36.8  65.    1. ]
  Symptômes  : [0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]
  Noms symp  : ['fracture'

## Sauvegarde

In [7]:
data_dir = Path('../data/models')
data_dir.mkdir(exist_ok=True)

# Pickle principal
output_path = data_dir / 'triage_dataset_v2.pkl'
with open(output_path, 'wb') as f:
    pickle.dump({
        'X': X,
        'y': y,
        'metadata': metadata_list,
        'feature_names': [
            'FC', 'FR', 'SpO2', 'TA_sys', 'TA_dia', 'Temp', 'Age', 'Sexe',
        ] + SYMPTOMES_CLES,
    }, f)

print(f'Dataset sauvegardé : {output_path}')

# CSV pour inspection
feature_names = ['FC', 'FR', 'SpO2', 'TA_sys', 'TA_dia', 'Temp', 'Age', 'Sexe'] + SYMPTOMES_CLES
df = pd.DataFrame(X, columns=feature_names)
df['label'] = y
csv_path = data_dir / 'triage_dataset_v2.csv'
df.to_csv(csv_path, index=False)
print(f'CSV sauvegardé    : {csv_path}')

Dataset sauvegardé : ..\data\models\triage_dataset_v2.pkl
CSV sauvegardé    : ..\data\models\triage_dataset_v2.csv
